In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

In [ ]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_SCALED = "BCCC17__scaled__v2"
NOMBRE_PCA = "BCCC17__pca__v2"

RUTA_SCALED = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SCALED
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_PCA

# ===== ARCHIVOS =====
TRAIN_FILE = f"{NOMBRE_SCALED}__train.csv"

PLOT_FILE = "pca_variance_plot.png"
REPORT_FILE = f"{NOMBRE_PCA}_report.json"

# ===== CONFIG =====
LABEL_COL = "LABEL"
VARIANCE_THRESHOLD = 0.95

In [ ]:
print("Ruta entrada:", RUTA_SCALED)
print("Ruta salida:", RUTA_SALIDA)

In [ ]:
train_path = RUTA_SCALED / TRAIN_FILE

if not train_path.exists():
    raise FileNotFoundError(train_path)

df = pd.read_csv(train_path)

print("Shape:", df.shape)

In [ ]:
X = df.drop(columns=[LABEL_COL])

print("Número de features:", X.shape[1])

In [ ]:
pca = PCA()
pca.fit(X)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

In [ ]:
k = np.argmax(cumulative_variance >= VARIANCE_THRESHOLD) + 1

print(f"k para {VARIANCE_THRESHOLD*100}% de varianza:", k)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(cumulative_variance, marker='o')
plt.axhline(y=VARIANCE_THRESHOLD, color='r', linestyle='--', label='95%')

plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada")
plt.title("PCA - Varianza explicada acumulada")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10, 6))
plt.plot(cumulative_variance)
plt.axhline(y=VARIANCE_THRESHOLD, linestyle='--')

plt.savefig(RUTA_SALIDA / PLOT_FILE)
plt.close()

In [ ]:
if k < 5:
    interpretacion = "Alta redundancia en las características"
elif k > 30:
    interpretacion = "Alta complejidad del espacio → justifica uso de LLM"
else:
    interpretacion = "Complejidad moderada del dataset"

print("Interpretación:", interpretacion)

In [ ]:
reporte = {
    "dataset": NOMBRE_SCALED,
    "num_features_original": int(X.shape[1]),
    "variance_threshold": VARIANCE_THRESHOLD,
    "k_components_95_variance": int(k),
    "explained_variance_first_10": explained_variance[:10].tolist(),
    "interpretation": interpretacion
}

with open(RUTA_SALIDA / REPORT_FILE, "w") as f:
    json.dump(reporte, f, indent=2)

print("Reporte guardado")

In [ ]:
print("========== PCA RESULT ==========")
print("Features originales:", X.shape[1])
print("k (95% varianza):", k)
print("Interpretación:", interpretacion)
print("===============================")

In [ ]:
print("========== ANÁLISIS DE COMPONENTES PCA ==========\n")

import pandas as pd

# Crear DataFrame con las componentes
componentes = pd.DataFrame(
    pca.components_,
    columns=X.columns
)

# Número de componentes a analizar (puedes cambiarlo)
NUM_COMPONENTES = 5

for i in range(NUM_COMPONENTES):
    print(f"--- COMPONENTE PRINCIPAL {i+1} ---")
    
    # Ordenar por importancia (valor absoluto)
    top_features = componentes.iloc[i].abs().sort_values(ascending=False)
    
    # Mostrar top 10
    print("Top 10 variables más importantes:")
    display(top_features.head(10))
    
    print("\nValores con signo (para interpretación):")
    display(componentes.iloc[i][top_features.head(10).index])
    
    print("\n")

print("===============================================")

In [ ]:
print("========== DIAGNÓSTICO PCA ==========\n")

# 1. Shape
print("Shape de X:", X.shape)
print()

# 2. Columnas
print("Número de columnas:", len(X.columns))
print("Primeras 20 columnas:")
print(X.columns[:20])
print()

# 3. ¿Está LABEL dentro?
print("¿LABEL está en X?:", "LABEL" in X.columns)
print()

# 4. Tipos de datos
print("Tipos de datos:")
print(X.dtypes.value_counts())
print()

# 5. Columnas no numéricas (MUY IMPORTANTE)
non_numeric = X.select_dtypes(exclude=["int64", "float64"]).columns
print("Columnas NO numéricas:", list(non_numeric))
print()

# 6. Estadísticas básicas
print("Estadísticas básicas (primeras columnas):")
print(X.describe().iloc[:, :10])
print()

# 7. Desviación estándar (detección de columnas constantes)
stds = X.std()
print("Columnas con std ≈ 0:")
print(stds[stds < 1e-6].sort_values())
print()

# 8. Varianza explicada PCA
from sklearn.decomposition import PCA
import numpy as np

pca = PCA()
pca.fit(X)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("Primeras 10 varianzas explicadas:")
print(explained[:10])
print()

print("Varianza acumulada primeras 10:")
print(cumulative[:10])
print()

# 9. Resultado k
k = np.argmax(cumulative >= 0.95) + 1
print("k calculado:", k)

print("\n====================================")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Proyección PCA
X_pca = pca.transform(X)

# Convertir labels a códigos numéricos
labels_cat = pd.Categorical(df[LABEL_COL])
labels_num = labels_cat.codes

# Opcional: muestrear para no pintar 2 millones de puntos
sample_size = 100000
if len(df) > sample_size:
    idx = np.random.RandomState(42).choice(len(df), sample_size, replace=False)
else:
    idx = np.arange(len(df))

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_pca[idx, 0],
    X_pca[idx, 1],
    c=labels_num[idx],
    s=1,
    cmap="tab20",
    alpha=0.6
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA 2D con clases")

# Leyenda
handles, _ = scatter.legend_elements()
plt.legend(
    handles,
    labels_cat.categories,
    title=LABEL_COL,
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    markerscale=6
)

plt.tight_layout()
plt.show()